## VIX Index Correlation V6
### Volume Acceleration Analysis

In [1]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm

from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy.stats import fisher_exact
from sklearn.metrics import roc_auc_score
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.request_api_v2 import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension
from Modules.fibonacci_retracement import FibonacciRetracement
from Modules.fibonacci_extension import FibonacciExtension

In [2]:
# FRED API key for accessing Federal Reserve Economic Data
FRED_API_KEY = os.getenv("FRED_API_KEY")
# Output directory for storing downloaded data
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
fred_url = "https://api.stlouisfed.org/fred/series/observations"
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()
# Fibonacciエクステンションをプロットするクラスのインスタンスを作成
fibonacci_extension_v2 = FibonacciExtension() 
# Fibonacciリトレースメントを計算するクラスのインスタンスを作成
fibonacci_retracement = FibonacciRetracement()

In [3]:
# Date range for fetching API s
start = '2010-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
frequency = "d"
aggregation_method = "avg"

In [4]:
def extract_csv(
    start: str,
    end: str,
    output_dir: str,
    file_name: str,
    csv_df: pd.DataFrame,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / file_name
    # 重複するファイルがある場合は削除
    if output_path.exists():
        output_path.unlink()
    # 保存
    df = csv_df.copy()
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]
    df.to_csv(output_path)
    return output_path

#### VIX時系列を取得

In [5]:
vix_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^VIX",
    market=None,
)
vix_df.tail()

取得件数: 4443


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4438,1340463,^VIX,None,2026-10-26,18.389999,19.840000,17.790001,19.660000,0,18.560,...,15.000243,47.139930,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4439,1376908,^VIX,None,2026-10-27,17.520000,18.549999,17.430000,18.430000,0,18.562,...,15.144815,47.596687,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4440,1376909,^VIX,None,2026-10-28,16.910000,17.770000,16.209999,17.469999,0,18.606,...,15.354685,47.554000,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4441,1376910,^VIX,None,2026-10-29,17.200001,17.360001,16.209999,17.150000,0,18.378,...,15.416081,50.644583,True,NaN,NaN,NaN,NaN,NaN,NaN,False
4442,1376911,^VIX,None,2026-10-30,19.420000,23.930000,18.030001,18.430000,0,18.228,...,15.509096,54.951763,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### S&P500時系列を取得

In [6]:
sp500_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^GSPC",
    market=None,
)
sp500_df.tail()

取得件数: 4441


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4436,1340449,^GSPC,None,2026-10-26,7676.770020,7702.129883,7639.700195,7639.700195,5838070000,7677.388086,...,7677.272749,14.983352,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4437,1372719,^GSPC,None,2026-10-27,7722.600098,7736.240234,7677.160156,7699.470215,5160120000,7681.480176,...,7674.871703,18.552036,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4438,1374591,^GSPC,None,2026-10-28,7743.979980,7758.430176,7718.370117,7735.560059,5466960000,7681.306152,...,7674.310572,27.456672,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4439,1374599,^GSPC,None,2026-10-29,7720.390137,7732.209961,7690.939941,7699.609863,5827600000,7682.304102,...,7676.715761,33.040212,False,NaN,NaN,NaN,NaN,1.666667,NaN,False
4440,1374607,^GSPC,None,2026-10-30,7580.069824,7637.479980,7527.560059,7631.069824,4268617000,7681.082031,...,7670.226174,28.634850,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### Nasdaq100時系列を取得

In [7]:
qqq_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="^IXIC",
    market=None,
)
qqq_df.tail()

取得件数: 4441


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
4436,1340464,^IXIC,None,2026-10-26,26169.380859,26306.820312,25986.560547,26026.869141,7402660000,26232.775781,...,26436.095556,33.159737,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4437,1372721,^IXIC,None,2026-10-27,26477.810547,26540.810547,26219.310547,26286.310547,7914230000,26223.453906,...,26389.878450,31.102194,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4438,1374595,^IXIC,None,2026-10-28,26549.849609,26587.339844,26414.919922,26541.609375,7985070000,26211.429688,...,26365.216353,34.022027,False,NaN,NaN,NaN,NaN,-50.0,NaN,True
4439,1374603,^IXIC,None,2026-10-29,26184.720703,26286.460938,26053.160156,26201.820312,7405880000,26190.111719,...,26328.123979,32.920686,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4440,1374609,^IXIC,None,2026-10-30,25436.998047,25672.998047,25177.330078,25592.486328,7710137000,26129.819141,...,26218.753156,25.382054,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### UVXY時系列を取得

In [8]:
uvxy_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="UVXY",
    market=None,
)
uvxy_df.tail()

取得件数: 3791


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
3786,1340465,UVXY,None,2026-10-26,18.090000,18.370001,17.629999,18.270000,8133200,17.928,...,16.9600,-73.093998,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3787,1374651,UVXY,None,2026-10-27,17.860001,18.160000,17.690001,17.900000,6631800,17.934,...,16.9916,-68.774012,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3788,1374652,UVXY,None,2026-10-28,17.330000,17.910000,17.180000,17.840000,7906000,18.128,...,17.0396,-64.240587,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3789,1374653,UVXY,None,2026-10-29,16.750000,17.520000,16.750000,17.440001,8604900,18.124,...,17.0456,-60.159652,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3790,1374654,UVXY,None,2026-10-30,19.330000,21.190001,18.070000,18.160000,21265300,17.922,...,17.0896,-54.456587,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### SVXY時系列を取得

In [9]:
svxy_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="SVXY",
    market=None,
)
svxy_df.tail()

取得件数: 3791


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
3786,1340466,SVXY,None,2026-10-26,62.220001,62.810001,62.009998,62.160000,1230300,62.636000,...,63.030542,67.425083,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3787,1374655,SVXY,None,2026-10-27,62.610001,62.799999,62.259998,62.540001,728500,62.606001,...,62.933147,61.307948,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3788,1374656,SVXY,None,2026-10-28,63.270000,63.400002,62.529999,62.639999,1399400,62.348000,...,62.832362,55.493896,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3789,1374657,SVXY,None,2026-10-29,63.930000,63.930000,63.020000,63.119999,1338400,62.322000,...,62.800253,50.755571,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3790,1374658,SVXY,None,2026-10-30,60.549999,62.209999,58.299999,62.099998,3982200,62.512000,...,62.673429,42.516862,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### VXX時系列を取得

In [10]:
vxx_df = request_api.fetch_stock_timeseries_data(
    start_date=start,
    end_date=end,
    code="VXX",
    market=None,
)
vxx_df.tail()

取得件数: 2204


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
2199,1340467,VXX,None,2026-10-26,18.309999,18.475000,17.980000,18.400000,8772000,18.128,...,16.914875,-70.135747,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2200,1374659,VXX,None,2026-10-27,18.120001,18.311001,18.000000,18.200001,6429400,18.152,...,16.921223,-65.363272,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2201,1374660,VXX,None,2026-10-28,17.790001,18.165001,17.674999,18.129999,9705700,18.298,...,16.947363,-60.309058,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2202,1374661,VXX,None,2026-10-29,17.410000,17.900000,17.370001,17.879999,10172600,18.320,...,16.953718,-55.929309,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2203,1374662,VXX,None,2026-10-30,19.209999,20.434999,18.280001,18.330000,39333500,18.188,...,16.964800,-49.278579,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### データ統合

In [11]:
# データフレームのコピーを作成
vix_work = vix_df.copy()
sp500_work = sp500_df.copy()
qqq_work = qqq_df.copy()
uvxy_work = uvxy_df.copy()
svxy_work = svxy_df.copy()
vxx_work = vxx_df.copy()
# 必要列のみ抽出
vix_work = vix_work[["date", "close"]].rename(
    columns={"close": "vix"}
)
# S&P500の必要列のみ抽出
sp500_work = sp500_work[["date", "close"]].rename(
    columns={"close": "sp500"}
)
# QQQの必要列のみ抽出
qqq_work = qqq_work[["date", "close"]].rename(
    columns={"close": "qqq"}
)
# UVXYの必要列のみ抽出
uvxy_work = uvxy_work[["date", "close", "volume"]].rename(
    columns={
        "close": "uvxy",
        "volume": "uvxy_volume"
    }
)
# SVXYの必要列のみ抽出
svxy_work = svxy_work[["date", "close", "volume"]].rename(
    columns={
        "close": "svxy",
        "volume": "svxy_volume"
    }
)
# VXXの必要列のみ抽出
vxx_work = vxx_work[["date", "close", "volume"]].rename(
    columns={
        "close": "vxx",
        "volume": "vxx_volume"
    }
)
# 結合用のデータフレームを作成
research_df = (
    vix_work
    .merge(sp500_work, on="date",how="left")
    .merge(qqq_work, on="date", how="left")
    .merge(uvxy_work, on="date", how="left")
    .merge(svxy_work, on="date", how="left")
    .merge(vxx_work, on="date", how="left")
)
# 結合結果の形状を確認
print(research_df.shape)
# CSVファイルとして抽出
extract_csv(
    start=start,
    end=end,
    output_dir=output_dir,
    file_name=f"research_df_{start}_{end}.csv",
    csv_df=research_df,
)
# データフレームの先頭を表示
display(research_df.head())

(4443, 10)


,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume
0,2009-03-09,49.349998,680.760010,1284.839966,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-03-10,49.680000,679.280029,1288.949951,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-03-11,43.349998,719.590027,1364.800049,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-03-12,43.529999,720.890015,1367.780029,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-03-13,40.990002,751.969971,1427.030029,NaN,NaN,NaN,NaN,NaN,NaN


#### 1. Volume ROC特徴量作成

##### 目的

出来高モメンタムを計算する。

- ROC5
- ROC10
- ROC20

In [12]:
# ボリューム列の変化率を計算
VOLUME_COLS = ["uvxy_volume", "svxy_volume", "vxx_volume"]
# 各ボリューム列の変化率を計算する日数のリスト
for volume_col in VOLUME_COLS:
    # 各日数に対して変化率を計算
    for n in [5, 10, 20]:
        # 変化率を計算して新しい列に格納
        research_df[f"{volume_col}_roc_{n}d"
        ] = (research_df[volume_col] - research_df[volume_col].shift(n))
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,uvxy_volume_roc_5d,uvxy_volume_roc_10d,uvxy_volume_roc_20d,svxy_volume_roc_5d,svxy_volume_roc_10d,svxy_volume_roc_20d,vxx_volume_roc_5d,vxx_volume_roc_10d,vxx_volume_roc_20d
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,-10746600.0,2813300.0,-320800.0,-2620000.0,417600.0,203300.0,-15444200.0,-858900.0,795800.0
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,-10583000.0,1533600.0,1149400.0,-1496800.0,-214300.0,-262700.0,-7174600.0,1598200.0,813200.0
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,-7016300.0,-6782400.0,2586200.0,-211700.0,-1226800.0,666800.0,-1140300.0,-3214600.0,3567600.0
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,-15329900.0,-4711400.0,1239900.0,-1771300.0,-333300.0,186400.0,-9685700.0,-1860300.0,1185300.0
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,1178900.0,-22704500.0,15625100.0,1994000.0,-9679800.0,3070500.0,23162300.0,-5096400.0,31917900.0


#### 2. Volume Change特徴量作成

##### 目的

N営業日前からの出来高変化率を計算する。

- Change5
- Change10
- Change20

In [13]:
# ボリューム列の変化率を計算
for volume_col in VOLUME_COLS:
    # 各日数に対して変化率を計算
    for n in [5, 10, 20]:
        # 過去n日分の値を取得し、0をNaNに置換
        base = (
            research_df[volume_col]
            .shift(n)
            .replace(0, np.nan)
        )
        # 変化率を計算して新しい列に格納
        research_df[f"{volume_col}_change_{n}d"] = (research_df[volume_col] / base - 1)
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_volume_roc_20d,uvxy_volume_change_5d,uvxy_volume_change_10d,uvxy_volume_change_20d,svxy_volume_change_5d,svxy_volume_change_10d,svxy_volume_change_20d,vxx_volume_change_5d,vxx_volume_change_10d,vxx_volume_change_20d
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,...,795800.0,-0.569212,0.528826,-0.037947,-0.680466,0.513843,0.197955,-0.637763,-0.089182,0.099772
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,...,813200.0,-0.614762,0.300812,0.209653,-0.672628,-0.227302,-0.265032,-0.527389,0.330808,0.144795
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,...,3567600.0,-0.470189,-0.461752,0.486146,-0.131401,-0.467139,0.910183,-0.105136,-0.248802,0.581222
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,...,1185300.0,-0.640486,-0.353807,0.168350,-0.569605,-0.199378,0.161806,-0.487741,-0.154601,0.131886
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,...,31917900.0,0.058691,-0.516366,2.770310,1.002917,-0.708520,3.367884,1.432318,-0.114707,4.304156


#### 3. Volume Acceleration特徴量作成

##### 目的

出来高モメンタムの変化速度を計算する。

Acceleration A

ROC5 - ROC10

Acceleration B

ROC10 - ROC20

In [14]:
# ボリューム列の加速度を計算
for volume_col in VOLUME_COLS:
    # 各ボリューム列に対して加速度を計算
    research_df[f"{volume_col}_accel_a"] = (
        research_df[f"{volume_col}_roc_5d"]
        -
        research_df[f"{volume_col}_roc_10d"]
    )
    # 各ボリューム列に対して加速度bを計算
    research_df[f"{volume_col}_accel_b"] = (
        research_df[f"{volume_col}_roc_10d"]
        -
        research_df[f"{volume_col}_roc_20d"]
    )
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,svxy_volume_change_20d,vxx_volume_change_5d,vxx_volume_change_10d,vxx_volume_change_20d,uvxy_volume_accel_a,uvxy_volume_accel_b,svxy_volume_accel_a,svxy_volume_accel_b,vxx_volume_accel_a,vxx_volume_accel_b
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,...,0.197955,-0.637763,-0.089182,0.099772,-13559900.0,3134100.0,-3037600.0,214300.0,-14585300.0,-1654700.0
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,...,-0.265032,-0.527389,0.330808,0.144795,-12116600.0,384200.0,-1282500.0,48400.0,-8772800.0,785000.0
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,...,0.910183,-0.105136,-0.248802,0.581222,-233900.0,-9368600.0,1015100.0,-1893600.0,2074300.0,-6782200.0
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,...,0.161806,-0.487741,-0.154601,0.131886,-10618500.0,-5951300.0,-1438000.0,-519700.0,-7825400.0,-3045600.0
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,...,3.367884,1.432318,-0.114707,4.304156,23883400.0,-38329600.0,11673800.0,-12750300.0,28258700.0,-37014300.0


#### 4. Volume Acceleration d(ROC)/dt特徴量作成

##### 目的

ROCの一次微分を計算する。

d(ROC)/dt

In [15]:
# ボリューム列のROC加速度を計算
for volume_col in VOLUME_COLS:
    # 各ボリューム列に対してROC加速度を計算
    research_df[f"{volume_col}_roc_acceleration"] = (
        research_df[f"{volume_col}_roc_10d"]
        -
        research_df[f"{volume_col}_roc_10d"].shift(1)
    )
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_volume_change_20d,uvxy_volume_accel_a,uvxy_volume_accel_b,svxy_volume_accel_a,svxy_volume_accel_b,vxx_volume_accel_a,vxx_volume_accel_b,uvxy_volume_roc_acceleration,svxy_volume_roc_acceleration,vxx_volume_roc_acceleration
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,...,0.099772,-13559900.0,3134100.0,-3037600.0,214300.0,-14585300.0,-1654700.0,-12877000.0,-773400.0,-10470100.0
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,...,0.144795,-12116600.0,384200.0,-1282500.0,48400.0,-8772800.0,785000.0,-1279700.0,-631900.0,2457100.0
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,...,0.581222,-233900.0,-9368600.0,1015100.0,-1893600.0,2074300.0,-6782200.0,-8316000.0,-1012500.0,-4812800.0
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,...,0.131886,-10618500.0,-5951300.0,-1438000.0,-519700.0,-7825400.0,-3045600.0,2071000.0,893500.0,1354300.0
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,...,4.304156,23883400.0,-38329600.0,11673800.0,-12750300.0,28258700.0,-37014300.0,-17993100.0,-9346500.0,-3236100.0


#### 5. Acceleration Z-Score作成

##### 目的

Acceleration異常度を標準化する。

In [16]:
# ボリューム列のROC加速度のZスコアを計算
for volume_col in VOLUME_COLS:
    # 各ボリューム列に対してROC加速度のZスコアを計算
    accel_col = (f"{volume_col}_roc_acceleration")
    mean_col = (
        research_df[accel_col]
        .rolling(60)
        .mean()
    )
    std_col = (
        research_df[accel_col]
        .rolling(60)
        .std()
        .replace(0, np.nan)
    )
    research_df[f"{accel_col}_z"] = (research_df[accel_col] - mean_col) / std_col
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,svxy_volume_accel_a,svxy_volume_accel_b,vxx_volume_accel_a,vxx_volume_accel_b,uvxy_volume_roc_acceleration,svxy_volume_roc_acceleration,vxx_volume_roc_acceleration,uvxy_volume_roc_acceleration_z,svxy_volume_roc_acceleration_z,vxx_volume_roc_acceleration_z
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,...,-3037600.0,214300.0,-14585300.0,-1654700.0,-12877000.0,-773400.0,-10470100.0,NaN,NaN,NaN
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,...,-1282500.0,48400.0,-8772800.0,785000.0,-1279700.0,-631900.0,2457100.0,NaN,NaN,NaN
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,...,1015100.0,-1893600.0,2074300.0,-6782200.0,-8316000.0,-1012500.0,-4812800.0,NaN,NaN,NaN
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,...,-1438000.0,-519700.0,-7825400.0,-3045600.0,2071000.0,893500.0,1354300.0,NaN,NaN,NaN
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,...,11673800.0,-12750300.0,28258700.0,-37014300.0,-17993100.0,-9346500.0,-3236100.0,NaN,NaN,NaN


#### 6. QQQ急落イベント作成

##### 目的

分類問題用ターゲットを作成する。

In [17]:
# QQQの1日下落を閾値ごとに計算
for threshold in [2, 3, 5]:
    research_df[f"qqq_1d_drop_{threshold}"] = (
        (research_df["qqq"].shift(-1) / research_df["qqq"] - 1) <= -(threshold / 100)
    ).astype(int)
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_volume_accel_b,uvxy_volume_roc_acceleration,svxy_volume_roc_acceleration,vxx_volume_roc_acceleration,uvxy_volume_roc_acceleration_z,svxy_volume_roc_acceleration_z,vxx_volume_roc_acceleration_z,qqq_1d_drop_2,qqq_1d_drop_3,qqq_1d_drop_5
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,...,-1654700.0,-12877000.0,-773400.0,-10470100.0,NaN,NaN,NaN,0,0,0
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,...,785000.0,-1279700.0,-631900.0,2457100.0,NaN,NaN,NaN,0,0,0
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,...,-6782200.0,-8316000.0,-1012500.0,-4812800.0,NaN,NaN,NaN,0,0,0
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,...,-3045600.0,2071000.0,893500.0,1354300.0,NaN,NaN,NaN,1,0,0
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,...,-37014300.0,-17993100.0,-9346500.0,-3236100.0,NaN,NaN,NaN,0,0,0


#### 7. VIX急騰イベント作成

##### 目的

VIX急騰ターゲットを作成する。

In [18]:
# VIXの1日リターンとスパイクを計算
research_df["vix_1d_return"] = (research_df["vix"].shift(-1) / research_df["vix"] - 1)
# VIXの1日リターンのスパイクを閾値ごとに計算
for threshold in [5, 10, 20]:
    research_df[f"vix_1d_spike_{threshold}"] = (
        research_df["vix_1d_return"] >= threshold / 100
    ).astype(int)
research_df.tail()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,uvxy_volume_roc_acceleration_z,svxy_volume_roc_acceleration_z,vxx_volume_roc_acceleration_z,qqq_1d_drop_2,qqq_1d_drop_3,qqq_1d_drop_5,vix_1d_return,vix_1d_spike_5,vix_1d_spike_10,vix_1d_spike_20
4438,2026-10-26,19.660000,7639.700195,26026.869141,18.270000,8133200.0,62.160000,1230300.0,18.400000,8772000.0,...,NaN,NaN,NaN,0,0,0,-0.062564,0,0,0
4439,2026-10-27,18.430000,7699.470215,26286.310547,17.900000,6631800.0,62.540001,728500.0,18.200001,6429400.0,...,NaN,NaN,NaN,0,0,0,-0.052089,0,0,0
4440,2026-10-28,17.469999,7735.560059,26541.609375,17.840000,7906000.0,62.639999,1399400.0,18.129999,9705700.0,...,NaN,NaN,NaN,0,0,0,-0.018317,0,0,0
4441,2026-10-29,17.150000,7699.609863,26201.820312,17.440001,8604900.0,63.119999,1338400.0,17.879999,10172600.0,...,NaN,NaN,NaN,1,0,0,0.074636,1,0,0
4442,2026-10-30,18.430000,7631.069824,25592.486328,18.160000,21265300.0,62.099998,3982200.0,18.330000,39333500.0,...,NaN,NaN,NaN,0,0,0,NaN,0,0,0


#### 8. ROC-AUC分析

##### 目的

AccelerationとROCの説明力を比較する。

In [19]:
# 特徴量とターゲットの組み合わせに対してROC AUCを算出する
FEATURES = [c for c in research_df.columns
    if any(key in c for key in [
            "_roc_",
            "_accel_",
            "_change_"
        ]
    )
]
# ターゲット列を定義する
TARGETS = [
    "qqq_1d_drop_2",
    "qqq_1d_drop_3",
    "qqq_1d_drop_5",
    "vix_1d_spike_5",
    "vix_1d_spike_10",
    "vix_1d_spike_20"
]
rows = []
# ROC AUCを計算するためのループを回す
for feature in FEATURES:
    # 各ターゲットに対してROC AUCを計算
    for target in TARGETS:
        # 特徴量とターゲットの組み合わせに対してデータを抽出
        work_df = (
            research_df[[feature, target]]
            .replace([np.inf, -np.inf], np.nan).dropna()
        )
        # ターゲット列のユニークな値が2未満の場合はスキップ
        if (work_df[target].nunique() < 2):
            continue
        # ROC AUCを計算
        auc = roc_auc_score(
            work_df[target],
            work_df[feature]
        )
        # 計算結果をリストに追加
        rows.append({
            "feature": feature,
            "target": target,
            "roc_auc": auc
        })
# 計算結果をDataFrameに変換
auc_df = pd.DataFrame(rows)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"8_roc_auc_analysis_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = auc_df.copy()
df.to_csv(output_path)
display(
    auc_df
    .sort_values("roc_auc", ascending=False).head(100)
)

,feature,target,roc_auc
26,svxy_volume_roc_10d,qqq_1d_drop_5,0.950275
20,svxy_volume_roc_5d,qqq_1d_drop_5,0.895309
80,svxy_volume_change_10d,qqq_1d_drop_5,0.886219
32,svxy_volume_roc_20d,qqq_1d_drop_5,0.881828
86,svxy_volume_change_20d,qqq_1d_drop_5,0.867604
...,...,...,...
16,uvxy_volume_roc_20d,vix_1d_spike_10,0.645308
174,vxx_volume_roc_acceleration_z,qqq_1d_drop_2,0.645244
4,uvxy_volume_roc_5d,vix_1d_spike_10,0.645240
148,uvxy_volume_roc_acceleration,vix_1d_spike_10,0.644467


#### 9. Top Feature抽出

##### 目的

Volume AccelerationがVolume ROCを上回るか確認する。

In [20]:
best_features_df = (
    auc_df
    .query("roc_auc >= 0.70")
    .sort_values("roc_auc", ascending=False)
)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"9_roc_auc_top_features_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = best_features_df.copy()
df.to_csv(output_path)
display(
    best_features_df
)

,feature,target,roc_auc
26,svxy_volume_roc_10d,qqq_1d_drop_5,0.950275
20,svxy_volume_roc_5d,qqq_1d_drop_5,0.895309
80,svxy_volume_change_10d,qqq_1d_drop_5,0.886219
32,svxy_volume_roc_20d,qqq_1d_drop_5,0.881828
86,svxy_volume_change_20d,qqq_1d_drop_5,0.867604
74,svxy_volume_change_5d,qqq_1d_drop_5,0.859793
29,svxy_volume_roc_10d,vix_1d_spike_20,0.783703
170,svxy_volume_roc_acceleration_z,qqq_1d_drop_5,0.765091
83,svxy_volume_change_10d,vix_1d_spike_20,0.764462
152,svxy_volume_roc_acceleration,qqq_1d_drop_5,0.763934


# VIX Index Correlation V6 - Task 1.1 結論

## 検証目的

V5で有効性が確認された Volume ROC に対して、

- Volume Change
- Volume Acceleration
- Acceleration Z-Score

がより早期かつ強力な市場ストレスシグナルとなるか検証した。

対象ETFは

- SVXY
- VXX
- UVXY

とし、

- QQQ翌日急落
- VIX翌日急騰

に対する説明力をROC-AUCで評価した。 【1-0bfd35】【2-5f5688】

---

## 検証結果

最も高いROC-AUCを示した特徴量は引き続き

- SVXY Volume ROC 10D

であった。

QQQ翌日-5%に対して

```text
ROC-AUC = 0.950
```

を記録し、V5の結果を再現した。 【1-0bfd35】【2-5f5688】

また、

- SVXY Volume ROC 5D
- SVXY Volume Change 10D
- SVXY Volume ROC 20D

も高い説明力を示した。 【1-0bfd35】

---

## Volume Acceleration評価

本研究の主仮説であった

```text
Volume Acceleration > Volume ROC
```

は支持されなかった。

代表的な特徴量である

- SVXY Volume ROC Acceleration
- SVXY Volume ROC Acceleration Z

のROC-AUCは

```text
約0.76
```

に留まり、

SVXY Volume ROC 10D の

```text
0.95
```

を大きく下回った。 【1-0bfd35】【2-5f5688】

---

## 解釈

市場ストレス検知において重要なのは

```text
出来高増加速度
```

ではなく、

```text
出来高モメンタムの蓄積量
```

である可能性が高い。

つまり、

市場参加者のヘッジ需要やリスクオフ行動は

Accelerationではなく

ROCとして観測されていると考えられる。 【1-0bfd35】【2-5f5688】

---

## 結論

Task 1.1 の結果から、

Volume Acceleration単独によるLater1シグナルの改善は確認できなかった。

現時点の最有力特徴量は引き続き

```text
SVXY Volume ROC 10D
```

である。

したがって、

Volume Acceleration研究は一旦終了し、

次フェーズである

```text
Task 2.1
Price × Volume 統合分析
```

へ移行する。

### Task 2.1
Price × Volume Analysis

#### 10. Price × Volume Analysis

##### 目的

価格変化と出来高フローを統合し、

市場参加者のリスクオフ行動を
より正確に捉えられるか検証する。

Task1.1で有効性が確認された
Volume ROCをベースラインとして比較する。

#### 11. Price Return特徴量作成

##### 目的

ETF価格変化率を計算する。

In [21]:
# 各価格列について、1日、5日、および10日間のリターンを算出する。
PRICE_COLS = ["uvxy", "svxy", "vxx"]
# 各価格列について、1日、5日、および10日間のリターンを算出する。
for price_col in PRICE_COLS:
    # 1日リターンを計算
    research_df[f"{price_col}_return_1d"] = (
        research_df[price_col]
        .pct_change()
    )
    # 5日リターンを計算
    research_df[f"{price_col}_return_5d"] = (
        research_df[price_col]
        .pct_change(5)
    )
    # 10日リターンを計算
    research_df[f"{price_col}_return_10d"] = (
        research_df[price_col]
        .pct_change(10)
    )
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"11_Price_Return_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
research_df.head()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vix_1d_spike_20,uvxy_return_1d,uvxy_return_5d,uvxy_return_10d,svxy_return_1d,svxy_return_5d,svxy_return_10d,vxx_return_1d,vxx_return_5d,vxx_return_10d
0,2009-03-09,49.349998,680.760010,1284.839966,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-03-10,49.680000,679.280029,1288.949951,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-03-11,43.349998,719.590027,1364.800049,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-03-12,43.529999,720.890015,1367.780029,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-03-13,40.990002,751.969971,1427.030029,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 12. Price Z-Score作成

##### 目的

価格変動の異常度を標準化する。

In [22]:
# 各価格列について、60日間のローリング平均と標準偏差を用いてZスコアを算出する
for price_col in PRICE_COLS:
    # 60日間のローリング平均を計算
    rolling_mean = (research_df[price_col].rolling(60).mean())
    # 60日間のローリング標準偏差を計算
    rolling_std = (
        research_df[price_col]
        .rolling(60)
        .std()
        .replace(0, np.nan)
    )
    # Zスコアを計算してデータフレームに追加
    research_df[f"{price_col}_z"] = (
        research_df[price_col] - rolling_mean
    ) / rolling_std
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"12_Price_Z_Score_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# データフレームの先頭を表示して確認
research_df.head()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,uvxy_return_10d,svxy_return_1d,svxy_return_5d,svxy_return_10d,vxx_return_1d,vxx_return_5d,vxx_return_10d,uvxy_z,svxy_z,vxx_z
0,2009-03-09,49.349998,680.760010,1284.839966,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-03-10,49.680000,679.280029,1288.949951,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-03-11,43.349998,719.590027,1364.800049,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-03-12,43.529999,720.890015,1367.780029,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-03-13,40.990002,751.969971,1427.030029,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 13. Price × Volume特徴量作成

##### 目的

価格下落と出来高増加の同時発生を特徴量化する。

In [23]:
# ETFごとの各種指標の計算（Return × ROC、Return × Change、Z × ROC）
ETF_LIST = ["uvxy", "svxy", "vxx"]
# 対象とするETFのリストを定義
for etf in ETF_LIST:
    # Return × ROC
    research_df[f"{etf}_return_x_roc"] = (
        # 10日間のリターンと出来高のROCを掛け合わせる
        research_df[f"{etf}_return_10d"] * research_df[f"{etf}_volume_roc_10d"]
    )
    # Return × Change
    research_df[f"{etf}_return_x_change"] = (
        # 10日間のリターンと出来高の変化を掛け合わせる
        research_df[f"{etf}_return_10d"]
        *
        research_df[f"{etf}_volume_change_10d"]
    )
    # Z × ROC
    research_df[f"{etf}_z_x_roc"] = (
        # Zスコアと出来高のROCを掛け合わせる
        research_df[f"{etf}_z"]
        *
        research_df[f"{etf}_volume_roc_10d"]
    )
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"13_Price_x_Volume_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# データフレームの先頭を表示して確認
research_df.head()

,date,vix,sp500,qqq,uvxy,uvxy_volume,svxy,svxy_volume,vxx,vxx_volume,...,vxx_z,uvxy_return_x_roc,uvxy_return_x_change,uvxy_z_x_roc,svxy_return_x_roc,svxy_return_x_change,svxy_z_x_roc,vxx_return_x_roc,vxx_return_x_change,vxx_z_x_roc
0,2009-03-09,49.349998,680.760010,1284.839966,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-03-10,49.680000,679.280029,1288.949951,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-03-11,43.349998,719.590027,1364.800049,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-03-12,43.529999,720.890015,1367.780029,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-03-13,40.990002,751.969971,1427.030029,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 14. Price × Volume ROC-AUC分析

##### 目的

複合特徴量が単独特徴量を上回るか確認する。

In [24]:
# Price × Volume の特徴量とターゲットの組み合わせに対するROC AUCを計算する
FEATURES = [c for c in research_df.columns
    if any(key in c for key in [
            "_return_x_roc",
            "_return_x_change",
            "_z_x_roc"
        ]
    )
]
# ターゲットのリストを定義
TARGETS = [
    "qqq_1d_drop_2",
    "qqq_1d_drop_3",
    "qqq_1d_drop_5",
    "vix_1d_spike_5",
    "vix_1d_spike_10",
    "vix_1d_spike_20"
]
# ROC AUCを計算するための行を格納するリストを初期化
rows = []
# 各特徴量とターゲットの組み合わせに対してROC AUCを計算
for feature in FEATURES:
    # 各ターゲットに対してROC AUCを計算
    for target in TARGETS:
        # 特徴量とターゲットのデータを抽出し、欠損値や無限大を処理
        work_df = (
            research_df[[feature, target]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )
        # ターゲットのユニークな値が2未満の場合はスキップ
        if work_df[target].nunique() < 2:
            continue
        # ROC AUCを計算
        auc = roc_auc_score(work_df[target], work_df[feature])
        # 計算結果を行に追加
        rows.append({
            "feature": feature,
            "target": target,
            "roc_auc": auc
        })
# 計算結果をDataFrameに変換
price_volume_auc_df = pd.DataFrame(rows)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"14_Price_x_Volume_ROC_AUC_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = price_volume_auc_df.copy()
df.to_csv(output_path)
# 上位100件のROC AUCを表示
display(price_volume_auc_df.sort_values("roc_auc", ascending=False).head(100))

,feature,target,roc_auc
8,uvxy_return_x_change,qqq_1d_drop_5,0.702210
38,vxx_return_x_roc,qqq_1d_drop_5,0.687030
44,vxx_return_x_change,qqq_1d_drop_5,0.670124
2,uvxy_return_x_roc,qqq_1d_drop_5,0.664037
50,vxx_z_x_roc,qqq_1d_drop_5,0.657094
14,uvxy_z_x_roc,qqq_1d_drop_5,0.642587
47,vxx_return_x_change,vix_1d_spike_20,0.567302
41,vxx_return_x_roc,vix_1d_spike_20,0.544258
5,uvxy_return_x_roc,vix_1d_spike_20,0.534402
11,uvxy_return_x_change,vix_1d_spike_20,0.533867


#### 15. ベンチマーク比較

##### 目的

V5・V6で有効だった
Volume ROCとの比較を行う。

In [25]:
# ベンチマーク特徴量に対するROC AUCを計算する
benchmark_features = ["svxy_volume_roc_10d", "svxy_volume_change_10d"]
benchmark_rows = []
# 各ベンチマーク特徴量とターゲットの組み合わせに対してROC AUCを計算する
for feature in benchmark_features:
    # 各ターゲットに対してROC AUCを計算
    for target in ["qqq_1d_drop_5", "vix_1d_spike_20"]:
        # 特徴量とターゲットのデータを抽出し、欠損値や無限大を処理
        work_df = (
            research_df[[feature, target]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )
        # ROC AUCを計算
        auc = roc_auc_score(work_df[target], work_df[feature])
        # 計算結果を辞書に追加
        benchmark_rows.append({
            "feature": feature,
            "target": target,
            "roc_auc": auc,
            "type": "benchmark"
        })
# 計算結果をDataFrameに変換
benchmark_df = pd.DataFrame(benchmark_rows)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"15_Benchmark_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = benchmark_df.copy()
df.to_csv(output_path)
# 計算結果を表示
display(benchmark_df)

,feature,target,roc_auc,type
0,svxy_volume_roc_10d,qqq_1d_drop_5,0.950275,benchmark
1,svxy_volume_roc_10d,vix_1d_spike_20,0.783703,benchmark
2,svxy_volume_change_10d,qqq_1d_drop_5,0.886219,benchmark
3,svxy_volume_change_10d,vix_1d_spike_20,0.764462,benchmark


#### 16. Top Feature抽出

##### 目的

Volume ROCを上回る
Price × Volume特徴量が存在するか確認する。

In [26]:
# ROC AUCが0.75以上の候補特徴量を抽出して表示する
candidate_df = (
    price_volume_auc_df
    .query("roc_auc >= 0.75")
    .sort_values("roc_auc", ascending=False)
)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"16_Candidate_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = candidate_df.copy()
df.to_csv(output_path)
# 計算結果を表示
display(candidate_df)

,feature,target,roc_auc


# VIX Index Correlation V6 - Task 2.1 Price × Volume Analysis

## 検証目的

Volume系特徴量に価格情報を統合することで、QQQ急落およびVIX急騰に対する説明力が向上するか検証した。

検証対象として以下の複合特徴量を作成した。

- Return × ROC
- Return × Change
- Z-Score × ROC

対象ETFは

- UVXY
- SVXY
- VXX

とした。 【1-2c9a72】

---

## 検証結果

Price × Volume 特徴量は、いずれも既存の Volume 特徴量を上回ることができなかった。 【1-2c9a72】【2-896368】

ベンチマークである

```text
SVXY Volume ROC 10D
```

は

```text
QQQ翌日-5%
ROC-AUC = 0.950
```

を維持した。 【2-896368】

一方で Price × Volume 特徴量の最高値は

```text
UVXY Return × Change
QQQ翌日-5%
ROC-AUC = 0.702
```

であり、ベンチマークを大きく下回った。 【1-2c9a72】

また、

```text
SVXY Return × ROC
QQQ翌日-5%
ROC-AUC = 0.120
```

```text
SVXY Z × ROC
QQQ翌日-5%
ROC-AUC = 0.112
```

となり、価格情報の追加によって大幅な性能低下が確認された。 【1-2c9a72】

---

## 考察

本検証結果は、

```text
価格情報の追加によって
シグナルが強化される
```

という仮説を支持しなかった。

むしろ、

```text
Volume単独の方が
市場ストレスをより純粋に捉えている
```

可能性が示唆された。 【2-896368】【1-2c9a72】

価格変動は Volume に含まれる市場参加者のフロー情報に対してノイズとなり、説明力を低下させる可能性がある。

---

## 結論

Task 2.1 の結果から、

```text
Price × Volume仮説
```

は棄却する。

Layer1は価格予測モデルではなく、

```text
市場参加者のフロー検知モデル
```

として整理する。

現時点でもっとも有効な特徴量は引き続き

```text
SVXY Volume ROC 10D
```

および

```text
SVXY_Z >= 3
AND
SVXY_ROC_10D Top Decile
```

である。

したがって次フェーズでは、新規特徴量探索を一旦停止し、

```text
Task 3
Event Study
```

へ進み、シグナル発生後に実際にどのようなリスクオフ現象が発生するかを検証する。